## Прогиб упругой мембраны ##

**1. Квадратная мембрана под равномерной нагрузкой**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# Параметры задачи
L = 1.0           # Длина стороны квадратной мембраны
N = 101           # Количество узлов по каждой оси
h = L / (N - 1)   # Шаг сетки
T = 1.0           # Постоянное натяжение мембраны
P = 1.0           # Равномерная нагрузка

In [ ]:
x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
X, Y = np.meshgrid(x, y)
u = np.zeros((N, N))

In [ ]:
# Граничные условия (u = 0 на границах)
u[0, :] = 0      # нижняя граница
u[-1, :] = 0     # верхняя граница
u[:, 0] = 0      # левая граница
u[:, -1] = 0     # правая граница

In [ ]:
# Правая часть уравнения Пуассона
f = -P / T * np.ones((N, N))

In [ ]:
max_iter = 10000
eps = 1e-6

In [ ]:
# Метод Гаусса-Зейделя
for iteration in range(max_iter):
    u_old = u.copy()
    u[1:-1, 1:-1] = 0.25 * (u[2:, 1:-1] + u[:-2, 1:-1] + u[1:-1, 2:] + u[1:-1, :-2] - h**2 * f[1:-1, 1:-1])
    
    error = np.max(np.abs(u - u_old))
    if error < eps:
        print(f"Сходимость достигнута на итерации {iteration+1}")
        break
    
    if iteration % 1000 == 0:
        print(f"Итерация {iteration}, ошибка: {error:.2e}")

In [ ]:
# Нахождение точки максимального прогиба
max_idx = np.unravel_index(np.argmax(u), u.shape)
max_u = u[max_idx]
max_x = x[max_idx[1]]
max_y = y[max_idx[0]]

print(f"\nМаксимальный прогиб: {max_u:.6f}")
print(f"Координаты точки максимального прогиба: x = {max_x:.3f}, y = {max_y:.3f}")

Визуализация:

In [ ]:
fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(X, Y, u, cmap='viridis', alpha=0.9)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('u(x,y)')
ax1.set_title('3D визуализация прогиба мембраны')
fig.colorbar(surf, ax=ax1, shrink=0.5)

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(15, 5))

ax2 = fig.add_subplot(122)
contour = ax2.contourf(X, Y, u, levels=50, cmap='viridis')
ax2.plot(max_x, max_y, 'ro', markersize=8, label=f'Макс. прогиб\n({max_x:.2f}, {max_y:.2f})')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Контурный график прогиба')
ax2.legend()
ax2.set_aspect('equal')
fig.colorbar(contour, ax=ax2, shrink=0.5)

plt.tight_layout()
plt.show()

**2. Мембрана сложной формы**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# Параметры задачи
L = 1.0           # Длина стороны
N = 150           # Количество узлов по каждой оси
h = L / (N - 1)   # Шаг сетки
T = 1.0           # Постоянное натяжение мембраны

Создаем L-образную область:

In [ ]:
def create_L_shape(N):
    mask = np.zeros((N, N), dtype=bool)
    mask[:, :N//2] = True
    mask[N//2:, :] = True
    return mask

Граничные точки:

In [ ]:
def create_boundary_mask(L_mask):
    
    boundary_mask = np.zeros_like(L_mask, dtype=bool)
    boundary_mask[0, :] = True
    boundary_mask[:N//2, 0] = True
    boundary_mask[:, -1] = True
    boundary_mask[-1, :N//2] = True
    boundary_mask[~L_mask] = False

    return boundary_mask

Локализованная нагрузка (гауссов пик):

In [ ]:
def gaussian_load(X, Y, center_x, center_y, sigma=0.1):
    return np.exp(-((X - center_x)**2 + (Y - center_y)**2) / (2 * sigma**2))

In [ ]:
x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
X, Y = np.meshgrid(x, y)

L_mask = create_L_shape(N)
boundary_mask = create_boundary_mask(L_mask)
u = np.zeros((N, N))

In [ ]:
# Граничные условия на внешних краях L-формы
u[boundary_mask] = 0

In [ ]:
load_center_x = 0.55 * L
load_center_y = 0.75 * L
P = gaussian_load(X, Y, load_center_x, load_center_y, sigma=0.05)
P[~L_mask] = 0

In [ ]:
# Правая часть уравнения Пуассона
f = -P / T

In [ ]:
max_iter = 15000
eps = 1e-6

In [ ]:
# Метод Гаусса-Зейделя для L-образной области
for iteration in range(max_iter):
    u_old = u.copy()
    
    interior_mask = L_mask & ~boundary_mask
    
    for i in range(1, N-1):
        for j in range(1, N-1):
            if interior_mask[i, j]:
                u[i, j] = 0.25 * (u[i+1, j] + u[i-1, j] + u[i, j+1] + u[i, j-1] - h**2 * f[i, j])
    
    domain_error = np.max(np.abs(u[L_mask] - u_old[L_mask]))
    if domain_error < eps:
        print(f"Сходимость достигнута на итерации {iteration+1}")
        break
    
    if iteration % 2000 == 0:
        print(f"Итерация {iteration}, ошибка: {domain_error:.2e}")

In [ ]:
u[~L_mask] = np.nan

In [ ]:
# Нахождение точки максимального прогиба внутри области
u_masked = u.copy()
u_masked[~L_mask] = -np.inf
max_idx = np.unravel_index(np.argmax(u_masked), u.shape)
max_u = u[max_idx]
max_x = x[max_idx[1]]
max_y = y[max_idx[0]]

In [ ]:
print(f"\nМаксимальный прогиб: {max_u:.6f}")
print(f"Координаты точки максимального прогиба: x = {max_x:.3f}, y = {max_y:.3f}")
print(f"Центр нагрузки: x = {load_center_x:.3f}, y = {load_center_y:.3f}")

In [ ]:
fig = plt.figure(figsize=(18, 12))

ax1 = fig.add_subplot(231, projection='3d')
surf = ax1.plot_surface(X, Y, u, cmap='viridis', alpha=0.9, linewidth=0, antialiased=True)
ax1.scatter([load_center_x], [load_center_y], [0], color='red', s=100, label='Центр нагрузки')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('u(x,y)')
ax1.set_title('3D визуализация прогиба L-образной мембраны')
ax1.legend()

ax2 = fig.add_subplot(232)
contour = ax2.contourf(X, Y, u, levels=50, cmap='viridis')
ax2.contour(X, Y, L_mask.astype(float), levels=[0.5], colors='black', linewidths=2)
ax2.plot(max_x, max_y, 'ro', markersize=8, label=f'Макс. прогиб\n({max_x:.2f}, {max_y:.2f})')
ax2.plot(load_center_x, load_center_y, 'rx', markersize=10, label='Центр нагрузки')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Контурный график прогиба')
ax2.legend()
ax2.set_aspect('equal')

ax3 = fig.add_subplot(233)
corner_x, corner_y = N//2, N//2
corner_region = u[corner_y-10:corner_y+10, corner_x-10:corner_x+10]
im = ax3.imshow(corner_region, cmap='viridis', extent=[x[corner_x-10], x[corner_x+10], y[corner_y-10], y[corner_y+10]])
ax3.axhline(y=y[corner_y], color='white', linestyle='-', alpha=0.5)
ax3.axvline(x=x[corner_x], color='white', linestyle='-', alpha=0.5)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('Область внутреннего угла')

ax4 = fig.add_subplot(234)
y_section_idx = np.argmin(np.abs(y - load_center_y))
x_section = u[y_section_idx, :]
ax4.plot(x, x_section, 'b-', linewidth=2, label=f'Сечение по x при y = {load_center_y:.2f}')
ax4.axvline(x=load_center_x, color='red', linestyle='--', alpha=0.7, label='Центр нагрузки')
ax4.set_xlabel('x')
ax4.set_ylabel('u(x, y=const)')
ax4.set_title('Сечение прогиба вдоль оси x')
ax4.legend()
ax4.grid(True)

ax5 = fig.add_subplot(235)
x_section_idx = np.argmin(np.abs(x - load_center_x))
y_section = u[:, x_section_idx]
ax5.plot(y, y_section, 'g-', linewidth=2, label=f'Сечение по y при x = {load_center_x:.2f}')
ax5.axvline(x=load_center_y, color='red', linestyle='--', alpha=0.7, label='Центр нагрузки')
ax5.set_xlabel('y')
ax5.set_ylabel('u(x=const, y)')
ax5.set_title('Сечение прогиба вдоль оси y')
ax5.legend()
ax5.grid(True)

plt.tight_layout()
plt.show()